# 87 — FNSPID external-review robustness pack

**Objective.** Execute the remaining computable diagnostics requested by the external panel: the exact prior-open interval, trailing-beta benchmark, FNSPID episode-drop and crash exclusion, contrast dependence checks, FinBERT class base rates, and stratum-specific realised rank spreads.

Every inferential item is post-review and post-selection. None is independent confirmation. Notebook 75 is hash-checked but not executed.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("run the notebook from inside the repository")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.lib.external_review_robustness import run  # noqa: E402

SPEC_PATH = ROOT / "experiments/specs/fnspid_external_review_robustness_pack_v1_20260816.json"
spec = json.loads(SPEC_PATH.read_text(encoding="utf-8"))
assert spec["status"] == "frozen_before_external_review_robustness_pack_results"
spec["frozen_at_utc"], spec["git_commit_at_freeze"]


('2026-08-16T14:01:35Z', 'acaf530c75a20e882166e7218245342384cf2b1b')

## Frozen plan

- Estimate the panel's exact prior-open-to-assigned-open probe with the baseline rank model.
- Replace fixed beta one with a trailing 252-session open-return beta (minimum 126 observations), then add beta rank as a control.
- Re-estimate the direct era contrast at HAC lags 0, 5, 21 and 42; test regime-demeaned residual dependence.
- Match the existing LSEG episode protocol on FNSPID, then exclude February–April 2020.
- Reconstruct only aggregate class counts from sufficient statistics and verify integer identities.
- Put each story-count stratum on its own realised rank-spread scale.


In [2]:
written = run()
tables = {
    name: pd.read_csv(path)
    for name, path in written.items()
    if path.suffix == ".csv"
}
manifest = json.loads(written["manifest"].read_text(encoding="utf-8"))
display(Markdown("## Input coverage"))
display(tables["coverage_audit"])


## Input coverage

,item,value
0,panel_rows,715546
1,panel_symbols,570
2,price_rows,1950672
3,rows_with_trailing_beta,714356
4,rows_with_prior_open_outcome,715525
5,beta_complete_rows,714345


## Prior-open and beta diagnostics

The prior-open row tests the panel report's exact interval. The beta rows are same-sample post-review robustness checks, with BH correction across the two new specifications.


In [3]:
display(tables["prior_open_timing_probe"])
display(Markdown("### Beta-adjusted coefficients"))
display(tables["beta_robustness_coefficients"])
display(Markdown("### Beta-adjusted era contrasts"))
display(tables["beta_robustness_contrasts"])


,test,role,coefficient,n_clusters,hac_lags,inference,estimate,se,t,p_two_sided,...,n_rows_complete,minimum_names,expected_direction,direction_matches,dates_total,dates_used,dates_below_min_names,dates_constant_variable,dates_rank_deficient,q_single_test
0,previous_open_to_assigned_open,single_post_review_timing_probe,beta_negative_share,2264,5,mean_daily_cross_sectional_rank_coefficient_hac,-0.019257,0.003337,-5.77126,7.868109e-09,...,512135,10,negative,True,2264,2264,0,0,0,7.868109e-09


### Beta-adjusted coefficients

,regime,specification,outcome,regressors,input_rows,multiplicity_family,coefficient,n_clusters,hac_lags,inference,...,p_two_sided,ci_low,ci_high,dates_total,dates_used,dates_below_min_names,dates_constant_variable,dates_rank_deficient,q_bh_two_beta_specs,bh_reject_q05
0,development,unit_beta_same_sample,ar_open_h1,"mean_continuous, negative_share, log1p_n",510960,same_sample_audit_excluded,beta_negative_share,2264,5,mean_daily_cross_sectional_rank_coefficient_hac,...,0.005018,-0.014145,-0.002511,2264,2264,0,0,0,NaN,False
1,development,trailing_beta_adjusted,ar_trailing_beta_open_h1,"mean_continuous, negative_share, log1p_n",510960,post_review_beta_family,beta_negative_share,2264,5,mean_daily_cross_sectional_rank_coefficient_hac,...,0.000417,-0.016399,-0.004687,2264,2264,0,0,0,0.000835,True
2,development,trailing_beta_adjusted_plus_beta_control,ar_trailing_beta_open_h1,"mean_continuous, negative_share, log1p_n, trai...",510960,post_review_beta_family,beta_negative_share,2264,5,mean_daily_cross_sectional_rank_coefficient_hac,...,0.001182,-0.015249,-0.003761,2264,2264,0,0,0,0.001182,True
3,evaluation,unit_beta_same_sample,ar_open_h1,"mean_continuous, negative_share, log1p_n",203385,same_sample_audit_excluded,beta_negative_share,998,5,mean_daily_cross_sectional_rank_coefficient_hac,...,0.257393,-0.004261,0.015928,998,998,0,0,0,NaN,False
4,evaluation,trailing_beta_adjusted,ar_trailing_beta_open_h1,"mean_continuous, negative_share, log1p_n",203385,post_review_beta_family,beta_negative_share,998,5,mean_daily_cross_sectional_rank_coefficient_hac,...,0.105647,-0.001753,0.018344,998,998,0,0,0,0.174247,False
5,evaluation,trailing_beta_adjusted_plus_beta_control,ar_trailing_beta_open_h1,"mean_continuous, negative_share, log1p_n, trai...",203385,post_review_beta_family,beta_negative_share,998,5,mean_daily_cross_sectional_rank_coefficient_hac,...,0.174247,-0.003000,0.016560,998,998,0,0,0,0.174247,False


### Beta-adjusted era contrasts

,specification,outcome,regressors,multiplicity_family,contrast,coefficient,reference,comparison,reference_estimate,comparison_estimate,...,p_two_sided,ci_low,ci_high,n_reference,n_comparison,n_clusters,hac_lags,inference,q_bh_two_beta_specs,bh_reject_q05
0,unit_beta_same_sample,ar_open_h1,"mean_continuous, negative_share, log1p_n",same_sample_audit_excluded,evaluation_minus_development,beta_negative_share,development,evaluation,-0.008328,0.005833,...,0.017273,0.002503,0.025819,2264,998,3262,5,daily_cross_sectional_rank_coefficient_regime_...,NaN,False
1,trailing_beta_adjusted,ar_trailing_beta_open_h1,"mean_continuous, negative_share, log1p_n",post_review_beta_family,evaluation_minus_development,beta_negative_share,development,evaluation,-0.010543,0.008296,...,0.001511,0.007200,0.030477,2264,998,3262,5,daily_cross_sectional_rank_coefficient_regime_...,0.003022,True
2,trailing_beta_adjusted_plus_beta_control,ar_trailing_beta_open_h1,"mean_continuous, negative_share, log1p_n, trai...",post_review_beta_family,evaluation_minus_development,beta_negative_share,development,evaluation,-0.009505,0.006780,...,0.004919,0.004936,0.027634,2264,998,3262,5,daily_cross_sectional_rank_coefficient_regime_...,0.004919,True


## Contrast dependence and episode robustness

Lag sensitivity is descriptive. The risk gate is deliberately strict: every leave-one-episode-out interval and the crash-excluded interval must remain above zero.


In [4]:
display(tables["contrast_hac_sensitivity"])
display(tables["coefficient_dependence_diagnostics"])
display(Markdown("### Episode attribution"))
display(tables["fnspid_episode_attribution"])
display(Markdown("### Leave-one-episode-out"))
display(tables["fnspid_leave_one_episode_out"])
display(Markdown("### February–April 2020 exclusion"))
display(tables["fnspid_crash_exclusion"])
display(Markdown(f"**Frozen episode gate:** `{manifest['episode_gate']}`"))


,contrast,coefficient,reference,comparison,reference_estimate,comparison_estimate,estimate,se,t,p_two_sided,ci_low,ci_high,n_reference,n_comparison,n_clusters,hac_lags,inference
0,evaluation_minus_development,beta_negative_share,development,evaluation,-0.00831,0.005833,0.014143,0.005931,2.384705,0.017093,0.002519,0.025768,2264,998,3262,0,daily_cross_sectional_rank_coefficient_regime_...
1,evaluation_minus_development,beta_negative_share,development,evaluation,-0.00831,0.005833,0.014143,0.005941,2.380604,0.017284,0.002499,0.025788,2264,998,3262,5,daily_cross_sectional_rank_coefficient_regime_...
2,evaluation_minus_development,beta_negative_share,development,evaluation,-0.00831,0.005833,0.014143,0.006212,2.276757,0.022801,0.001968,0.026319,2264,998,3262,21,daily_cross_sectional_rank_coefficient_regime_...
3,evaluation_minus_development,beta_negative_share,development,evaluation,-0.00831,0.005833,0.014143,0.006084,2.324774,0.020084,0.002219,0.026067,2264,998,3262,42,daily_cross_sectional_rank_coefficient_regime_...


,diagnostic,lag,statistic,p_value
0,acf,1,0.005369,NaN
1,ljung_box,5,6.413174,0.268065
2,ljung_box,10,8.035077,0.625411
3,ljung_box,21,26.390025,0.191966


### Episode attribution

,episode,start,end,n_sessions,return_diff_sum,downside_reduction_sum,downside_share
0,1,2020-01-28,2020-01-29,2,-0.004808,-1.162408e-06,-0.000227
1,2,2020-02-25,2020-03-03,6,0.018625,7.316422e-04,0.143050
2,3,2020-03-06,2020-03-25,14,0.033158,1.537666e-03,0.300644
3,4,2020-03-30,2020-04-03,5,0.003236,1.440004e-04,0.028155
4,5,2020-04-14,2020-04-20,5,-0.000752,1.751587e-05,0.003425
5,6,2020-04-22,2020-04-23,2,-0.002038,0.000000e+00,0.000000
6,7,2020-06-09,2020-06-15,5,0.024131,4.390184e-04,0.085837
7,8,2020-09-22,2020-09-23,2,-0.002458,-1.695920e-06,-0.000332
8,9,2021-03-24,2021-03-26,3,0.001292,1.101436e-04,0.021535
9,10,2021-07-20,2021-07-21,2,-0.008442,0.000000e+00,0.000000


### Leave-one-episode-out

,dropped_episode,start,end,n,mean,ci_low,ci_high,p_two_sided,block_length,replications,seed,ci_excludes_zero
0,1,2020-01-28,2020-01-29,998,0.000005,0.000002,0.000010,0.0140,20,4999,20260830,True
1,2,2020-02-25,2020-03-03,998,0.000004,0.000002,0.000008,0.0116,20,4999,20260831,True
2,3,2020-03-06,2020-03-25,998,0.000004,0.000002,0.000006,0.0028,20,4999,20260832,True
3,4,2020-03-30,2020-04-03,998,0.000005,0.000002,0.000009,0.0156,20,4999,20260833,True
4,5,2020-04-14,2020-04-20,998,0.000005,0.000002,0.000010,0.0162,20,4999,20260834,True
5,6,2020-04-22,2020-04-23,998,0.000005,0.000002,0.000010,0.0152,20,4999,20260835,True
6,7,2020-06-09,2020-06-15,998,0.000005,0.000002,0.000009,0.0234,20,4999,20260836,True
7,8,2020-09-22,2020-09-23,998,0.000005,0.000002,0.000010,0.0176,20,4999,20260837,True
8,9,2021-03-24,2021-03-26,998,0.000005,0.000002,0.000010,0.0180,20,4999,20260838,True
9,10,2021-07-20,2021-07-21,998,0.000005,0.000002,0.000010,0.0140,20,4999,20260839,True


### February–April 2020 exclusion

,excluded_start,excluded_end,excluded_sessions,n,mean,ci_low,ci_high,p_two_sided,block_length,replications,seed,ci_excludes_zero
0,2020-02-01,2020-04-30,62,998,0.000003,0.000001,0.000004,0.0034,20,4999,20260930,True


**Frozen episode gate:** `{'episodes': 44, 'leave_one_out_all_intervals_above_zero': True, 'crash_excluded_interval_above_zero': True, 'passes': True}`

## Measurement audit

Class rates are aggregate arithmetic, not label validation. Stratum translations use each subsample's own attainable rank spread and remain descriptive.


In [5]:
display(tables["finbert_class_base_rates"])
display(tables["negative_share_tie_mass"])
display(tables["story_count_stratum_rank_spreads"])
display(
    Markdown(
        "Carry the results into the aggregate snapshot, evidence map and manuscript "
        "without describing any row as independent confirmation, causal timing, factor alpha "
        "or external label validation."
    )
)


,regime,class,story_count,story_share,total_stories
0,development,negative,198851,0.167954,1183961
1,development,neutral,710470,0.600079,1183961
2,development,positive,274640,0.231967,1183961
3,evaluation,negative,65821,0.158418,415490
4,evaluation,neutral,255193,0.614198,415490
5,evaluation,positive,94476,0.227385,415490
6,overall,negative,264672,0.165477,1599451
7,overall,neutral,965663,0.603747,1599451
8,overall,positive,369116,0.230777,1599451


,regime,firm_days,negative_share_equals_zero,negative_share_equals_zero_fraction,negative_share_equals_one,negative_share_equals_one_fraction,singleton_fraction
0,development,512153,383758,0.749303,43026,0.084010,0.483195
1,evaluation,203393,156301,0.768468,17512,0.086099,0.565251
2,overall,715546,540059,0.754751,60538,0.084604,0.506519


,stratum,model_rows,sessions,coefficient,pooled_rank_p10,pooled_rank_p90,pooled_rank_spread,pooled_fitted_percentile_points,mean_session_rank_spread,mean_session_fitted_percentile_points,count_control_included
0,n = 1,247395,2263,-0.010252,-0.089474,0.413934,0.503408,-0.516106,0.355568,-0.364536,False
1,n >= 2,264679,2264,-0.005291,-0.212195,0.399286,0.611481,-0.323552,0.565410,-0.299175,True
2,n >= 3,137120,2262,-0.001828,-0.261538,0.406250,0.667788,-0.122098,0.623355,-0.113974,True


Carry the results into the aggregate snapshot, evidence map and manuscript without describing any row as independent confirmation, causal timing, factor alpha or external label validation.